# 11 — 特徴点検出・マッチング

## 目標
- ORB / AKAZE で特徴点を検出・記述
- BFMatcher でブルートフォースマッチング
- Lowe's ratio test で誤対応を除去

## アルゴリズム比較
| 手法 | 記述子 | 速度 | 精度 | ライセンス |
|------|-------|------|------|----------|
| ORB | Binary (256bit) | 速 | 中 | BSD |
| AKAZE | Binary (486bit) | 中 | 高 | Apache |
| SIFT | Float (128次元) | 遅 | 高 | BSD (contrib) |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import cv2
import numpy as np
from utils import DATA_DIR, load_image, show_nb

img1 = load_image(DATA_DIR / 'lena.png')
# img2: 30° 回転させた画像で自己マッチング
h, w = img1.shape[:2]
M = cv2.getRotationMatrix2D((w/2, h/2), 30, 0.9)
img2 = cv2.warpAffine(img1, M, (w, h))
show_nb([('img1', img1), ('img2 (rotated 30°)', img2)], cols=2)

In [ ]:
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

# ORB
orb = cv2.ORB_create(nfeatures=500)
kp1, des1 = orb.detectAndCompute(gray1, None)
kp2, des2 = orb.detectAndCompute(gray2, None)

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = sorted(bf.match(des1, des2), key=lambda m: m.distance)
orb_match = cv2.drawMatches(img1, kp1, img2, kp2, matches[:40], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
print(f'ORB: kp1={len(kp1)}, kp2={len(kp2)}, matches={len(matches)}')
show_nb([('ORB matches (top 40)', orb_match)], cols=1)

In [ ]:
# AKAZE
akaze = cv2.AKAZE_create()
akp1, ades1 = akaze.detectAndCompute(gray1, None)
akp2, ades2 = akaze.detectAndCompute(gray2, None)

bf2 = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
amatches = sorted(bf2.match(ades1, ades2), key=lambda m: m.distance)
akaze_match = cv2.drawMatches(img1, akp1, img2, akp2, amatches[:40], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
print(f'AKAZE: kp1={len(akp1)}, kp2={len(akp2)}, matches={len(amatches)}')
show_nb([('AKAZE matches (top 40)', akaze_match)], cols=1)